# VEX Override: fix the camera labels

Our first detector drew boxes around game objects in about 50 pictures per batch. Many boxes are wrong or missing.
Your job is to **fix** them, not to draw everything from scratch. About 1–1.5 hours.

**One-time setup (2 min):** Colab can only see folders in your *My Drive*, not *Shared with me*. In Google Drive, find
the **label_batches** folder (the one with batch_1.zip … batch_4.zip and this notebook), right-click it →
*Organize* → **Add shortcut** → *My Drive*.

**Then:** run cell 1, pick your batch in cell 2 and run it, run cell 3 and start labelling.
Your work saves to Drive after every picture, so you can stop and come back any time: it continues where you left off.


In [ ]:
#@title 1. Setup (run once; click "Connect to Google Drive" when asked)
!pip install -q jupyter-bbox-widget
from google.colab import drive, output
output.enable_custom_widget_manager()
drive.mount("/content/drive", force_remount=True)

import glob, os
# the folder that holds batch_1.zip, up to 4 folders deep in My Drive (shortcuts count) or a shared drive
found = [os.path.dirname(p) for root in ("/content/drive/MyDrive", "/content/drive/Shareddrives")
         for depth in range(5) for p in glob.glob(root + "/*" * depth + "/batch_1.zip")]
FOLDER = found[0] if found else None
assert FOLDER, ("Can't find the batch files. Add a shortcut to the label_batches folder in My Drive "
                "(see the top of this notebook), then run this cell again.")
print("Found the batches in", FOLDER)


In [ ]:
#@title 2. Pick your batch, then run this cell
BATCH = "batch_1" #@param ["batch_1", "batch_2", "batch_3", "batch_4"]
import zipfile
batch_dir = f"{FOLDER}/{BATCH}"
if not os.path.isdir(f"{batch_dir}/images"):
    zipfile.ZipFile(f"{FOLDER}/{BATCH}.zip").extractall(batch_dir)
print(BATCH, ":", len(glob.glob(f"{batch_dir}/images/*.jpg")), "pictures")


## How to fix a picture
Click on the picture first so it reacts to the keys.

| To do this | Do this |
|---|---|
| Pick the class for new boxes | press its number: 1 yellow part, 2 blue part, 3 red part, 4 red goal, 5 blue goal, 6 robot, 7 person, 8 pin, 9 neutral goal, 0 cup. **Toggles and loaders have no key: click the class name** |
| Add a missing object | drag a box on the picture (it gets the selected class) |
| Select a box | click it, or `Tab` / `Shift+Tab` to step through the boxes |
| Delete the selected box | `Delete` (**on a Mac: `fn` + `delete`**) |
| Change the selected box's class | press the class number, then `C` |
| Move / resize the selected box | drag it or its corners |
| **Save and go to the next picture** | `Enter` or the **Submit** button |
| Next picture without saving | `Space` or **Skip** |

**Check every object in the picture, not only the boxes already drawn.** The model missed a lot,
and it never drew whole pins, neutral goals, cups, toggles or loaders, so you add those yourself.
**Example pictures** of correct boxes: `labelling_example.png`, `labelling_example_2.png`, `labelling_example_3_pins.png` (whole pins), `labelling_example_4_toggle.png` (toggles) and `how_to_label_a_stack.png` in the same Drive folder.

## What each class means (from the Override game manual v2.1)
| Class | What to box |
|---|---|
| **pin** | **one box around each whole pin** (both halves together; for a hidden pin, around the part you can see). This tells the robot which halves belong to the same pin, for picking it up |
| **yellow / blue / red part** | **one box per pin half, inside the pin box.** Every pin has two halves (red/yellow, blue/yellow, red/blue or yellow/yellow), and each half scores on its own. So a red/yellow pin lying on the floor gets **three boxes**: pin + red part + yellow part. If the pin lies diagonally, the boxes overlap: that's fine |
| **red goal / blue goal** | the **short red or blue octagonal goal** only (the body, not the pins on it) |
| **neutral goal** | the **black octagonal goals** (medium ones in the quadrants, tall one in the centre); the body only |
| **cup** | the whole hourglass cup (top half clear, bottom half dark grey): one box |
| **red / blue / yellow toggle** | the long triangular bar on top of the field wall: **one box around the whole bar**. Class = the colour of its **top face** (the manual's pictures of the 3 states, page 22). You often also see a second colour underneath: that is just the next side. The pairs are fixed: **yellow over red = yellow toggle, blue over yellow = blue toggle, red over blue = red toggle.** Can't tell: don't box it |
| **red / blue loader** | the corner loader: clear tube on a black frame on the wall, with a red or blue cap and base. **One box around the whole loader. Also box the pins and cups inside it** like any other piece (pin + halves, cup): the robot picks them from the bottom opening, and knows they are in the loader because their boxes are inside the loader box |
| **robot** | any robot (box the whole robot) |
| **person** | any person you can see |

**Fix these common model mistakes:**
- **"red goal" / "blue goal" boxes on pins** (a pin close to the camera looks big): change them to **red part / blue part**,
  split into two boxes if the pin has two colours.
- "red goal" on the toggle or a loader: change to the right **toggle** / **loader** class.
- **Green "pin" boxes are automatic guesses** (made from two touching halves). Check each one holds **exactly one pin**:
  two pins lying side by side often get merged into one box. Fix or delete those, and add pin boxes the guess missed.- Boxes on tape on the floor, shadows, reflections, banners or the wall: delete.

### Team rules (so all four of us label the same way)
- **Stacks:** no special class: box **each pin** (+ its visible halves), **each cup** and the **goal** separately.
- **Pin with a cup on it:** a **cup** box + a **pin** box around the part of the pin you can see + a box for each visible half.
  Seen through the **clear** half of the cup counts as visible; under the **dark** half does not. Pin completely hidden: cup box only.
  A pin half hidden inside a goal or under the dark half of a cup is not visible, so it gets no box.
- **Half-hidden objects:** label them if roughly half or more is visible.
- **Outside the field wall:** don't label game objects outside the field (spare pins, cups). Do label **people** and **robots** anywhere.
- **Tiny, far-away objects:** skip anything you can't identify with confidence.
- A picture with nothing in it: delete all boxes and press `Enter` (that saves "nothing here").


In [ ]:
#@title 3. Label (run, then click on the picture to start)
import datetime, glob, os
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from jupyter_bbox_widget import BBoxWidget

# Only ever ADD classes at the end (never reorder or delete), so saved boxes keep their meaning, and bump
# CLASSES_VERSION. Every save is logged with the version, so pictures saved before a class existed can be re-checked.
#   v1 (2026-10-28): 0-6 match the pre-drawn boxes; 7-13 added from the Override manual v2.1
#   v2 (2026-10-29): "pin" (whole pin, for picking) inserted at 7 so it gets key 8; v1's 7-13 moved to 8-14.
#                    Allowed only because no one had labelled yet; a v1-saved picture needs 7-13 -> 8-14.
CLASSES_VERSION = 2
classes = ["yellow part", "blue part", "red part", "red goal", "blue goal", "robot", "person",
           "pin", "neutral goal", "cup", "red toggle", "blue toggle", "yellow toggle", "red loader", "blue loader"]
colors = ["#ffd400", "#1f6fff", "#ff2020", "#8b0000", "#00008b", "#9467bd", "#ff7f0e",
          "#2ca02c", "#7f7f7f", "#17becf", "#e377c2", "#7ec8ff", "#bcbd22", "#c71585", "#4169e1"]
images = sorted(glob.glob(f"{batch_dir}/images/*.jpg"))
fixed_dir = f"{batch_dir}/fixed_labels"
os.makedirs(fixed_dir, exist_ok=True)
stem = lambda p: os.path.splitext(os.path.basename(p))[0]
is_done = lambda p: os.path.exists(f"{fixed_dir}/{stem(p)}.txt")
sizes = {}  # picture sizes, read once, so saving never has to re-open the picture
size = lambda p: sizes[p] if p in sizes else sizes.setdefault(p, Image.open(p).size)


def load_boxes(p):
    # fixed boxes if this picture was already saved, otherwise the model's boxes
    W, H = size(p)
    txt = f"{fixed_dir}/{stem(p)}.txt" if is_done(p) else f"{batch_dir}/labels/{stem(p)}.txt"
    boxes = []
    for line in (open(txt).read().split("\n") if os.path.exists(txt) else []):
        if line.strip():
            c, cx, cy, w, h = line.split()
            cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
            boxes.append({"x": cx - w / 2, "y": cy - h / 2, "width": w, "height": h, "label": classes[int(c)]})
    return boxes if is_done(p) else boxes + suggest_pins(boxes)


PIN_PAIRS = {("red part", "yellow part"), ("blue part", "yellow part"), ("blue part", "red part"), ("yellow part", "yellow part")}


def suggest_pins(boxes):
    # the model drew pin halves but never whole pins: pair touching halves of a colour pair that exists,
    # closest pairs first, and suggest a pin box around each pair (people check them like any other box)
    halves = [b for b in boxes if b["label"] in ("yellow part", "blue part", "red part")]
    pairs = []
    for i, a in enumerate(halves):
        for b in halves[i + 1:]:
            if tuple(sorted((a["label"], b["label"]))) not in PIN_PAIRS:
                continue
            gap = max(max(a["x"], b["x"]) - min(a["x"] + a["width"], b["x"] + b["width"]),
                      max(a["y"], b["y"]) - min(a["y"] + a["height"], b["y"] + b["height"]))
            size = min((a["width"] * a["height"]) ** 0.5, (b["width"] * b["height"]) ** 0.5)
            ratio = (a["width"] * a["height"]) / (b["width"] * b["height"])
            if gap <= 0.15 * size and 0.2 <= ratio <= 5:
                pairs.append((gap / size, id(a), id(b), a, b))
    used, pins = set(), []
    for _, ia, ib, a, b in sorted(pairs, key=lambda t: t[0]):
        if ia in used or ib in used:
            continue
        used |= {ia, ib}
        x0, y0 = min(a["x"], b["x"]), min(a["y"], b["y"])
        x1, y1 = max(a["x"] + a["width"], b["x"] + b["width"]), max(a["y"] + a["height"], b["y"] + b["height"])
        pins.append({"x": x0, "y": y0, "width": x1 - x0, "height": y1 - y0, "label": "pin"})
    return pins


def save_boxes(p, boxes):
    W, H = size(p)
    lines = []
    for b in boxes:
        x0, y0 = max(0.0, b["x"]), max(0.0, b["y"])
        x1, y1 = min(float(W), b["x"] + b["width"]), min(float(H), b["y"] + b["height"])
        if x1 - x0 < 2 or y1 - y0 < 2 or b.get("label") not in classes:
            continue
        lines.append(f"{classes.index(b['label'])} {(x0 + x1) / 2 / W:.6f} {(y0 + y1) / 2 / H:.6f} "
                     f"{(x1 - x0) / W:.6f} {(y1 - y0) / H:.6f}")
    open(f"{fixed_dir}/{stem(p)}.txt", "w").write("\n".join(lines) + "\n" * bool(lines))
    with open(f"{fixed_dir}/save_log.csv", "a") as log:
        log.write(f"{stem(p)},{CLASSES_VERSION},{datetime.datetime.now().isoformat(timespec='seconds')}\n")


i = next((k for k, p in enumerate(images) if not is_done(p)), 0)
box_widget = BBoxWidget(classes=classes, colors=colors, image=images[i], bboxes=load_boxes(images[i]))
status = widgets.HTML()
back = widgets.Button(description="◀ Previous picture")


DRIVE_LOST = ("<b style='color:#c00'>Google Drive disconnected, so this picture was NOT saved.</b> "
              "Run cell 1 again, then cells 2 and 3: it continues at this picture. "
              "All pictures saved before are safe.")


def show(k):
    global i
    try:
        boxes = load_boxes(images[k])
    except OSError:
        status.value = DRIVE_LOST
        return
    i = k
    box_widget.bboxes = boxes
    box_widget.image = images[i]
    saved = sum(is_done(p) for p in images)
    status.value = (f"<b>{BATCH}</b> · picture {i + 1} of {len(images)} ({stem(images[i])}) · "
                    f"saved {saved}/{len(images)}" + (" · <b>this one is already saved</b>" if is_done(images[i]) else ""))


@box_widget.on_submit
def _():
    try:
        save_boxes(images[i], box_widget.bboxes)
    except OSError:
        status.value = DRIVE_LOST
        return
    if all(is_done(p) for p in images):
        status.value = f"<b>All {len(images)} pictures saved. Thank you!</b> Run cell 4 to finish."
        return
    show(next((k for k in list(range(i + 1, len(images))) + list(range(i)) if not is_done(images[k])), i))


@box_widget.on_skip
def _():
    show(min(i + 1, len(images) - 1))


back.on_click(lambda _: show(max(i - 1, 0)))
show(i)
display(widgets.VBox([status, back, box_widget]))


In [ ]:
#@title 4. Finish (run when you are done or taking a break: makes sure everything is uploaded to Drive)
drive.flush_and_unmount()
print("Saved to Drive. You can close this tab.")
